# 🎙️ TTS Benchmark — Hindi & English Open-Source Models

This notebook benchmarks multiple open-source **Text-to-Speech (TTS)** systems across Hindi and English, producing a full suite of performance, quality, prosody, and linguistic robustness metrics.  It is the **TTS leg** of a Hindi ↔ English Speech-to-Speech translation pipeline (complementing the ASR and MT benchmarks).

---

## 🤖 Models Benchmarked

| Model | Language(s) | Architecture | Source |
|---|---|---|---|
| **MMS-TTS** (ENG / HIN) | English + Hindi | VITS end-to-end | Meta / HuggingFace |
| **SpeechT5** | English | Transformer + HiFi-GAN | Microsoft / HuggingFace |
| **Parler-TTS Mini** | English | Decoder-only (style-conditioned) | HuggingFace |
| **Coqui VITS** (EN / HI) | English + Hindi | VITS (language-specific) | Coqui-AI |
| **XTTS v2** | English + Hindi | Zero-shot multilingual | Coqui-AI |

---

## 📊 Metrics

| Category | Metric | Direction |
|---|---|---|
| **Performance** | Latency (ms) | ↓ lower is better |
| **Performance** | Real-Time Factor (RTF) | ↓ lower is better (<1 = faster than real-time) |
| **Performance** | Throughput (chars/sec) | ↑ higher is better |
| **Quality** | MOS — UTMOS neural predictor [1–5] | ↑ higher is better |
| **Quality** | Intelligibility — WER % (Whisper ASR) | ↓ lower is better |
| **Quality** | Intelligibility — CER % (Whisper ASR) | ↓ lower is better |
| **Prosody** | Pitch mean / std / range (Hz) | context-dependent |
| **Prosody** | Speaking rate (onsets/sec) | context-dependent |
| **Prosody** | Energy dynamics (RMS std) | ↑ higher = more expressive |
| **Prosody** | Pause ratio | context-dependent |
| **Robustness** | Per-category WER across 8 linguistic types | ↓ lower is better |

---

## 📁 Outputs
- `tts_benchmark_results/csv/` — 4 CSVs (full, summary, robustness, per-model)
- `tts_benchmark_results/plots/` — 11 publication-quality PNG plots
- `tts_benchmark_results/audio/` — all synthesised WAV files

## 1. Install Dependencies

The cell below auto-detects the environment (**Kaggle** or **local**) and Python version, then installs all packages.

Key dependencies:
- `transformers` — MMS-TTS, SpeechT5, Parler-TTS
- `TTS` (Coqui) — VITS EN/HI and XTTS v2
- `openai-whisper` + `jiwer` — intelligibility scoring (WER / CER)
- `librosa` — prosody extraction (pitch, energy, onsets)
- `utmos` — neural MOS prediction _(optional; MOS will be NaN if absent)_
- `soundfile`, `pandas`, `matplotlib`, `seaborn` — I/O and visualisation

### ⚠️ Coqui TTS & Python version compatibility

| Python version | Coqui TTS install strategy |
|---|---|
| **< 3.12** | `pip install TTS>=0.22.0` (official PyPI release) |
| **≥ 3.12** | Installs from the **[idiap/coqui-ai-TTS](https://github.com/idiap/coqui-ai-TTS)** community fork which supports 3.12+ |

If Coqui fails entirely, **Coqui-VITS and XTTS-v2 are skipped gracefully** — the remaining three models (MMS-TTS, SpeechT5, Parler-TTS) still run.

> ⚠️ **First run:** model weights are downloaded from HuggingFace Hub on demand (~200 MB – 2 GB per model). Subsequent runs use the local cache.

In [ ]:
import subprocess, sys, os, shutil

ON_KAGGLE = os.path.exists("/kaggle/working")
PY_VER    = sys.version_info
print(f"Environment : {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Python      : {PY_VER.major}.{PY_VER.minor}.{PY_VER.micro}")
print()

# ─────────────────────────────────────────────────────────────────────────────
def pip(*pkgs, fatal=True):
    """Install packages; if fatal=False, catches errors and returns False."""
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "--no-warn-conflicts", *pkgs]
    if fatal:
        subprocess.check_call(cmd)
        return True
    try:
        subprocess.check_call(cmd)
        return True
    except subprocess.CalledProcessError:
        return False

# ── Core ML (always required) ─────────────────────────────────────────────────
print("[1/6] Installing core ML packages …")
pip("transformers>=4.40.0", "accelerate>=0.27.0",
    "datasets>=2.18.0",     "sentencepiece>=0.1.99")
print("  ✓ transformers / accelerate / datasets")

# ── Coqui TTS — Python-version-aware install ──────────────────────────────────
# Official PyPI release caps at Python <3.12. For 3.12+ we use the
# community-maintained idiap fork which has identical import paths.
print("[2/6] Installing Coqui TTS …")
COQUI_AVAILABLE = False

if PY_VER < (3, 12):
    # Official release works fine on Python 3.9 – 3.11
    ok = pip("TTS>=0.22.0", fatal=False)
    if ok:
        COQUI_AVAILABLE = True
        print("  ✓ Coqui TTS (official PyPI) installed")
    else:
        print("  ⚠ Official TTS failed — trying idiap community fork …")
        ok = pip("coqui-tts", fatal=False)
        if ok:
            COQUI_AVAILABLE = True
            print("  ✓ Coqui TTS (coqui-tts fork) installed")
else:
    # Python 3.12+: official PyPI release does NOT support this version.
    # idiap/coqui-ai-TTS is the actively-maintained fork with 3.12 support.
    print(f"  Python {PY_VER.major}.{PY_VER.minor} detected — official TTS PyPI release "
          "does not support Python ≥3.12.")
    print("  Trying idiap/coqui-ai-TTS community fork …")
    ok = pip("coqui-tts", fatal=False)
    if ok:
        COQUI_AVAILABLE = True
        print("  ✓ coqui-tts (idiap fork) installed")
    else:
        print("  Trying git install of idiap fork …")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--no-warn-conflicts",
                "git+https://github.com/idiap/coqui-ai-TTS.git"
            ])
            COQUI_AVAILABLE = True
            print("  ✓ Coqui TTS installed from idiap git")
        except Exception as e:
            print(f"  ✗ Coqui TTS unavailable for Python {PY_VER.major}.{PY_VER.minor}: {e}")
            print("  → Coqui-VITS and XTTS-v2 models will be SKIPPED (non-fatal).")

# ── Parler-TTS ───────────────────────────────────────────────────────────────
print("[3/6] Installing Parler-TTS …")
PARLER_AVAILABLE = pip("parler-tts", fatal=False)
if not PARLER_AVAILABLE:
    # Fall back to installing from HuggingFace source
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "git+https://github.com/huggingface/parler-tts.git"
        ])
        PARLER_AVAILABLE = True
        print("  ✓ Parler-TTS installed from git")
    except Exception:
        print("  ⚠ Parler-TTS unavailable — Parler-TTS model will be SKIPPED (non-fatal).")
if PARLER_AVAILABLE:
    print("  ✓ parler-tts")

# ── Audio I/O & prosody ───────────────────────────────────────────────────────
print("[4/6] Installing audio / prosody packages …")
pip("soundfile>=0.12.1", "librosa>=0.10.1", "scipy>=1.12.0")
print("  ✓ soundfile / librosa / scipy")

# ── Whisper (intelligibility / WER) ──────────────────────────────────────────
print("[5/6] Installing Whisper + jiwer …")
pip("openai-whisper>=20231117", "jiwer>=3.0.3")
print("  ✓ openai-whisper / jiwer")

# ── Data / visualisation ──────────────────────────────────────────────────────
print("[6/6] Installing data / viz packages …")
pip("pandas>=2.1.0", "matplotlib>=3.8.0", "seaborn>=0.13.0", "numpy>=1.24.0")
print("  ✓ pandas / matplotlib / seaborn / numpy")

# ── UTMOS MOS predictor (optional) ───────────────────────────────────────────
print("[opt] Installing UTMOS MOS predictor …")
UTMOS_AVAILABLE = pip("utmos", fatal=False)
if UTMOS_AVAILABLE:
    print("  ✓ utmos — MOS scoring enabled")
else:
    print("  ⚠ utmos not available — MOS column will be NaN (non-fatal)")

# ── ffmpeg (required by Whisper) ──────────────────────────────────────────────
if shutil.which("ffmpeg") is None:
    if ON_KAGGLE:
        os.system("apt-get install -y ffmpeg -q")
        print("  ✓ ffmpeg installed via apt")
    else:
        print("  ⚠ ffmpeg not found.")
        print("    Linux : sudo apt install ffmpeg")
        print("    macOS : brew install ffmpeg")
        print("    Windows: https://ffmpeg.org/download.html")
else:
    print("  ✓ ffmpeg found")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print("=" * 50)
print("  INSTALL SUMMARY")
print("=" * 50)
print(f"  Coqui TTS (VITS / XTTS-v2) : {'✓ available' if COQUI_AVAILABLE   else '✗ skipped (Python version)'}")
print(f"  Parler-TTS                  : {'✓ available' if PARLER_AVAILABLE  else '✗ skipped'}")
print(f"  UTMOS MOS scorer            : {'✓ available' if UTMOS_AVAILABLE   else '⚠ NaN (optional)'}")
print("=" * 50)

## 2. Imports & Global Configuration

In [ ]:
import logging, os, sys, time, warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | Device: {DEVICE.upper()}")

### 2.1 Logging Setup

Logs are written both to stdout (visible in the notebook) and to `tts_benchmark.log` for post-run inspection.

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("tts_benchmark.log", encoding="utf-8"),
    ],
)
log = logging.getLogger(__name__)

### 2.2 Output Directories

All artefacts are written under `tts_benchmark_results/`.  On Kaggle the directory is created inside `/kaggle/working/`.

In [ ]:
ON_KAGGLE  = os.path.exists("/kaggle/working")
BASE_DIR   = Path("/kaggle/working") if ON_KAGGLE else Path(".")

OUTPUT_DIR = BASE_DIR / "tts_benchmark_results"
AUDIO_DIR  = OUTPUT_DIR / "audio"
PLOT_DIR   = OUTPUT_DIR / "plots"
CSV_DIR    = OUTPUT_DIR / "csv"

for _d in [OUTPUT_DIR, AUDIO_DIR, PLOT_DIR, CSV_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print(f"Output root : {OUTPUT_DIR.resolve()}")
print(f"Audio       : {AUDIO_DIR}")
print(f"Plots       : {PLOT_DIR}")
print(f"CSVs        : {CSV_DIR}")

### 2.3 Benchmark Hyperparameters

| Parameter | Default | Description |
|---|---|---|
| `N_WARMUP_RUNS` | 1 | Discarded runs (eliminates JIT / cache cold-start bias) |
| `N_TIMED_RUNS` | 3 | Measured runs; latency = mean wall-clock time |
| `SKIP_MOS` | False | Set `True` to skip UTMOS (faster, MOS column → NaN) |
| `SKIP_WHISPER` | False | Set `True` to skip WER/CER scoring |
| `LANGUAGES` | ["en", "hi"] | Languages to benchmark — change to `["en"]` for English only |

In [ ]:
# ── ✏️  Edit these to customise the benchmark run ─────────────────────────────
N_WARMUP_RUNS = 1        # warm-up iterations (discarded)
N_TIMED_RUNS  = 3        # measured iterations

LANGUAGES     = ["en", "hi"]   # "en", "hi", or both

SKIP_MOS      = False    # True → skip UTMOS MOS prediction
SKIP_WHISPER  = False    # True → skip Whisper WER/CER scoring

# Which models to run (set to None to run ALL)
# Choices: "MMS-TTS", "SpeechT5", "Parler-TTS", "Coqui-VITS", "XTTS-v2"
RUN_MODELS    = None     # e.g. ["MMS-TTS", "SpeechT5"] for a quick test
# ─────────────────────────────────────────────────────────────────────────────

print(f"Languages   : {LANGUAGES}")
print(f"Warmup runs : {N_WARMUP_RUNS}  |  Timed runs: {N_TIMED_RUNS}")
print(f"Skip MOS    : {SKIP_MOS}")
print(f"Skip Whisper: {SKIP_WHISPER}")
print(f"Models      : {RUN_MODELS or 'ALL'}")

## 3. Test Corpus — Linguistic Challenge Categories

Each language has **8 sentence categories** designed to stress-test different TTS capabilities:

| Category | Tests |
|---|---|
| `short` | Basic synthesis quality on minimal input |
| `medium` | Typical conversational sentence |
| `long` | Coherence and prosody over longer utterances |
| `numbers` | Number/date/time rendering |
| `named_entities` | Proper noun pronunciation |
| `technical` | Jargon, abbreviations, units |
| `punctuation` | Handling of dashes, ellipses, question marks |
| `abbreviations` (EN) / `mixed_script` (HI) | Abbreviation expansion / Devanagari–ASCII mixing |

In [ ]:
EN_CORPUS: Dict[str, str] = {
    "short":          "Hello, how are you today?",
    "medium":         "The quick brown fox jumps over the lazy dog.",
    "long": (
        "India is a remarkably diverse country with many languages, cultures, "
        "and traditions that have evolved over thousands of years of rich and "
        "complex history."
    ),
    "numbers": (
        "Call me at 9876543210 on the 15th of August 2024 at half past three "
        "in the afternoon."
    ),
    "named_entities": (
        "Prime Minister Narendra Modi met President Biden in New Delhi to "
        "discuss bilateral relations and trade agreements."
    ),
    "technical": (
        "The neural network operates at 3.5 gigahertz with 16 gigabytes of "
        "RAM and a 512-core GPU accelerator."
    ),
    "punctuation": (
        "Wait — are you serious?  I can't believe it!  Well, that's... "
        "truly unexpected."
    ),
    "abbreviations": (
        "Dr. Smith from MIT visited NASA's JPL facility in Los Angeles, CA, "
        "last Tuesday afternoon."
    ),
}

HI_CORPUS: Dict[str, str] = {
    "short":          "नमस्ते, आप कैसे हैं?",
    "medium":         "भारत एक विविधताओं से भरा हुआ देश है।",
    "long": (
        "भारत एक ऐसा महान देश है जहाँ अनेक भाषाएँ, संस्कृतियाँ और "
        "परंपराएँ हजारों वर्षों से निरंतर विकसित होती आई हैं।"
    ),
    "numbers": (
        "मुझे पाँच किलो चावल, तीन किलो दाल और दो लीटर सरसों का तेल "
        "चाहिए।"
    ),
    "named_entities": (
        "प्रधानमंत्री नरेंद्र मोदी ने नई दिल्ली में राष्ट्रपति भवन में "
        "एक महत्वपूर्ण बैठक आयोजित की।"
    ),
    "technical": (
        "यह कंप्यूटर 3.5 गीगाहर्ट्ज़ की गति से कार्य करता है और इसमें "
        "सोलह गीगाबाइट की मेमोरी है।"
    ),
    "punctuation": (
        "रुकिए — क्या आप सच कह रहे हैं?  मुझे बिल्कुल विश्वास नहीं होता!"
    ),
    "mixed_script": (
        "मेरा फ़ोन नंबर है 9876543210 और ईमेल पता है example@gmail.com।"
    ),
}

CORPORA: Dict[str, Dict[str, str]] = {"en": EN_CORPUS, "hi": HI_CORPUS}

print(f"English corpus : {len(EN_CORPUS)} sentences")
print(f"Hindi corpus   : {len(HI_CORPUS)} sentences")
print(f"Categories     : {list(EN_CORPUS.keys())}")

## 4. Result Data-Class

Every synthesised utterance produces a `TTSResult` — a typed record holding all measured metrics.  These are later concatenated into a single Pandas DataFrame.

In [ ]:
@dataclass
class TTSResult:
    # ── Identity ──────────────────────────────────────────────────────────────
    model_name      : str   = ""
    language        : str   = ""
    category        : str   = ""
    text            : str   = ""
    audio_path      : str   = ""
    # ── Performance ───────────────────────────────────────────────────────────
    latency_ms      : float = float("nan")  # wall-clock synthesis time (ms)
    rtf             : float = float("nan")  # synthesis_time / audio_duration
    throughput_cps  : float = float("nan")  # characters synthesised per second
    audio_duration_s: float = float("nan")  # length of generated audio (s)
    # ── Quality ───────────────────────────────────────────────────────────────
    mos_utmos       : float = float("nan")  # UTMOS MOS prediction [1–5]
    wer             : float = float("nan")  # Whisper WER (%)
    cer             : float = float("nan")  # Whisper CER (%)
    # ── Prosody ───────────────────────────────────────────────────────────────
    pitch_mean_hz   : float = float("nan")  # mean voiced F0
    pitch_std_hz    : float = float("nan")  # pitch std dev → naturalness
    pitch_range_hz  : float = float("nan")  # max F0 − min F0
    speaking_rate   : float = float("nan")  # onset events / second
    energy_std      : float = float("nan")  # RMS energy std → expressiveness
    pause_ratio     : float = float("nan")  # fraction of frames that are silent
    # ── Error ─────────────────────────────────────────────────────────────────
    error           : str   = ""

print("TTSResult dataclass defined.")
print("Fields:", [f.name for f in TTSResult.__dataclass_fields__.values()])

## 5. TTS Model Wrappers

Each model is wrapped in a class with three methods:
- **`load()`** — download weights and build the model
- **`synthesize(text, lang, out_path)`** — generate audio, write WAV, return duration
- **`unload()`** — free GPU/CPU memory before the next model loads

This pattern keeps peak memory low and matches the style used in the MT benchmark.

### 5.1 Base Class

In [ ]:
class BaseTTS:
    """Abstract base — all TTS backends inherit from this."""
    name: str = "base"
    supported_langs: List[str] = []

    def load(self):   raise NotImplementedError
    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        """Returns audio duration in seconds."""
        raise NotImplementedError
    def unload(self):
        torch.cuda.empty_cache()

print("BaseTTS defined.")

### 5.2 Model 1 — MMS-TTS (Meta, English + Hindi)

[facebook/mms-tts-eng](https://huggingface.co/facebook/mms-tts-eng) and [facebook/mms-tts-hin](https://huggingface.co/facebook/mms-tts-hin) are VITS models from Meta's Massively Multilingual Speech project.  They support 1,100+ languages and are lightweight enough to run on CPU.

In [ ]:
class MMSTTSWrapper(BaseTTS):
    name = "MMS-TTS"
    supported_langs = ["en", "hi"]
    _LANG_TO_HF_ID = {
        "en": "facebook/mms-tts-eng",
        "hi": "facebook/mms-tts-hin",
    }

    def __init__(self):
        self._models: Dict[str, Tuple] = {}

    def load(self):
        from transformers import VitsModel, VitsTokenizer
        import soundfile  # verify available
        for lang, model_id in self._LANG_TO_HF_ID.items():
            if lang not in LANGUAGES:
                continue
            log.info(f"  [{self.name}] Loading {model_id} …")
            tok   = VitsTokenizer.from_pretrained(model_id)
            model = VitsModel.from_pretrained(model_id).to(DEVICE)
            model.eval()
            self._models[lang] = (model, tok)
        log.info(f"  [{self.name}] Ready for {list(self._models.keys())}")

    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        import soundfile as sf
        model, tok = self._models[lang]
        inputs = tok(text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model(**inputs)
        wav = out.waveform.squeeze().cpu().numpy()
        sr  = model.config.sampling_rate
        sf.write(str(out_path), wav, sr)
        return len(wav) / sr

    def unload(self):
        del self._models
        super().unload()

print("MMSTTSWrapper defined.")

### 5.3 Model 2 — SpeechT5 (Microsoft, English)

[microsoft/speecht5_tts](https://huggingface.co/microsoft/speecht5_tts) is a Transformer encoder-decoder TTS with a HiFi-GAN neural vocoder.  Speaker identity is controlled via an x-vector embedding drawn from the CMU-Arctic dataset (speaker #7306 = SLT female voice by default).

In [ ]:
class SpeechT5Wrapper(BaseTTS):
    name = "SpeechT5"
    supported_langs = ["en"]

    def load(self):
        from transformers import (
            SpeechT5ForTextToSpeech, SpeechT5HifiGan, SpeechT5Processor
        )
        from datasets import load_dataset as _hf_ds
        log.info(f"  [{self.name}] Loading model + HiFi-GAN vocoder …")
        self.proc    = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
        self.model   = SpeechT5ForTextToSpeech.from_pretrained(
            "microsoft/speecht5_tts"
        ).to(DEVICE)
        self.vocoder = SpeechT5HifiGan.from_pretrained(
            "microsoft/speecht5-hifigan"
        ).to(DEVICE)
        self.model.eval()
        self.vocoder.eval()

        log.info(f"  [{self.name}] Loading speaker embedding (CMU-Arctic SLT) …")
        emb_ds = _hf_ds("Matthijs/cmu-arctic-xvectors", split="validation")
        self.spk_emb = torch.tensor(
            emb_ds[7306]["xvector"]
        ).unsqueeze(0).to(DEVICE)
        log.info(f"  [{self.name}] Ready.")

    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        import soundfile as sf
        inputs = self.proc(text=text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            speech = self.model.generate_speech(
                inputs["input_ids"], self.spk_emb, vocoder=self.vocoder
            )
        wav = speech.cpu().numpy()
        sr  = 16_000
        sf.write(str(out_path), wav, sr)
        return len(wav) / sr

    def unload(self):
        del self.model, self.proc, self.vocoder, self.spk_emb
        super().unload()

print("SpeechT5Wrapper defined.")

### 5.4 Model 3 — Parler-TTS Mini (HuggingFace, English)

[parler-tts/parler-tts-mini-v1](https://huggingface.co/parler-tts/parler-tts-mini-v1) is a decoder-only TTS conditioned on a **natural-language voice description**.  Edit `VOICE_DESC` below to change voice style (gender, speed, accent, quality).

In [ ]:
class ParlerTTSWrapper(BaseTTS):
    name = "Parler-TTS"
    supported_langs = ["en"]

    # ✏️ Edit this description to change the synthesised voice style
    VOICE_DESC = (
        "A female speaker delivers a slightly expressive and animated speech "
        "with a moderate speed and pitch. The recording is of very high "
        "quality, with the speaker's voice sounding clear and very close up."
    )

    def load(self):
        try:
            from parler_tts import ParlerTTSForConditionalGeneration
        except ImportError:
            raise RuntimeError("parler-tts not installed: pip install parler-tts")
        from transformers import AutoTokenizer
        log.info(f"  [{self.name}] Loading parler-tts-mini-v1 …")
        self.model = ParlerTTSForConditionalGeneration.from_pretrained(
            "parler-tts/parler-tts-mini-v1"
        ).to(DEVICE)
        self.tok = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-v1")
        self.model.eval()
        log.info(f"  [{self.name}] Ready.")

    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        import soundfile as sf
        desc_tok   = self.tok(self.VOICE_DESC, return_tensors="pt").to(DEVICE)
        prompt_tok = self.tok(text,            return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            gen = self.model.generate(
                input_ids             = desc_tok.input_ids,
                attention_mask        = desc_tok.attention_mask,
                prompt_input_ids      = prompt_tok.input_ids,
                prompt_attention_mask = prompt_tok.attention_mask,
            )
        wav = gen.cpu().numpy().squeeze()
        sr  = self.model.config.sampling_rate
        sf.write(str(out_path), wav, sr)
        return len(wav) / sr

    def unload(self):
        del self.model, self.tok
        super().unload()

print("ParlerTTSWrapper defined.")

### 5.5 Model 4 — Coqui VITS (English + Hindi)

Language-specific VITS models from Coqui TTS:
- English: `tts_models/en/ljspeech/vits` (trained on LJSpeech)
- Hindi: `tts_models/hi/cv/vits` (trained on CommonVoice)

These are single-speaker models optimised for each language independently.

In [ ]:
class CoquiVITSWrapper(BaseTTS):
    name = "Coqui-VITS"
    supported_langs = ["en", "hi"]
    _LANG_TO_MODEL = {
        "en": "tts_models/en/ljspeech/vits",
        "hi": "tts_models/hi/cv/vits",
    }

    def __init__(self):
        self._instances: Dict[str, object] = {}

    def load(self):
        try:
            from TTS.api import TTS as _CoquiTTS
        except ImportError:
            raise RuntimeError("Coqui TTS not installed: pip install TTS")
        for lang, model_id in self._LANG_TO_MODEL.items():
            if lang not in LANGUAGES:
                continue
            log.info(f"  [{self.name}] Loading {model_id} …")
            self._instances[lang] = _CoquiTTS(model_id, gpu=(DEVICE == "cuda"))
        log.info(f"  [{self.name}] Ready for {list(self._instances.keys())}")

    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        import soundfile as sf
        self._instances[lang].tts_to_file(text=text, file_path=str(out_path))
        data, sr = sf.read(str(out_path))
        return len(data) / sr

    def unload(self):
        del self._instances
        super().unload()

print("CoquiVITSWrapper defined.")

### 5.6 Model 5 — XTTS v2 (Coqui, multilingual zero-shot)

[XTTS v2](https://huggingface.co/coqui/XTTS-v2) is Coqui's flagship zero-shot multilingual TTS.  It supports 17 languages (including Hindi and English) from a single model using built-in studio speakers — no reference audio required.

In [ ]:
class XTTSv2Wrapper(BaseTTS):
    name = "XTTS-v2"
    supported_langs = ["en", "hi"]
    _LANG_CODES = {"en": "en", "hi": "hi"}
    _MODEL_ID   = "tts_models/multilingual/multi-dataset/xtts_v2"

    def load(self):
        try:
            from TTS.api import TTS as _CoquiTTS
        except ImportError:
            raise RuntimeError("Coqui TTS not installed: pip install TTS")
        log.info(f"  [{self.name}] Loading XTTS v2 (multilingual) …")
        self.tts = _CoquiTTS(self._MODEL_ID, gpu=(DEVICE == "cuda"))
        self._speaker = (
            self.tts.speakers[0] if self.tts.speakers else "Claribel Dervla"
        )
        log.info(f"  [{self.name}] Using speaker: {self._speaker}")
        log.info(f"  [{self.name}] Ready.")

    def synthesize(self, text: str, lang: str, out_path: Path) -> float:
        import soundfile as sf
        self.tts.tts_to_file(
            text      = text,
            speaker   = self._speaker,
            language  = self._LANG_CODES.get(lang, "en"),
            file_path = str(out_path),
        )
        data, sr = sf.read(str(out_path))
        return len(data) / sr

    def unload(self):
        del self.tts
        super().unload()

print("XTTSv2Wrapper defined.")

### 5.7 Model Registry

In [ ]:
ALL_MODELS: List[BaseTTS] = [
    MMSTTSWrapper(),
    SpeechT5Wrapper(),
    ParlerTTSWrapper(),
    CoquiVITSWrapper(),
    XTTSv2Wrapper(),
]
MODEL_REGISTRY: Dict[str, BaseTTS] = {m.name: m for m in ALL_MODELS}

print("Model registry:")
for name, m in MODEL_REGISTRY.items():
    print(f"  {name:15s}  →  supports: {m.supported_langs}")

## 6. Metric Computation

Each metric function is self-contained and gracefully returns `NaN` if its dependency is unavailable.

### 6.1 Prosody Extraction (librosa)

Uses `librosa.pyin` for robust probabilistic fundamental frequency (F0) estimation.  
Energy computed via per-frame RMS; onset density used as a tempo / speaking-rate proxy.

In [ ]:
def compute_prosody(wav_path: Path) -> Dict[str, float]:
    """
    Extract prosodic features from a WAV file.

    Returns
    -------
    dict with keys:
      pitch_mean_hz   – mean voiced F0 (Hz)
      pitch_std_hz    – pitch standard deviation (higher = more varied)
      pitch_range_hz  – max − min F0 in voiced frames
      speaking_rate   – onset events per second (proxy for syllable rate)
      energy_std      – RMS energy std dev (proxy for expressiveness)
      pause_ratio     – fraction of frames with RMS < 0.01 (silence)
    """
    nan_dict = {k: float("nan") for k in
                ["pitch_mean_hz", "pitch_std_hz", "pitch_range_hz",
                 "speaking_rate", "energy_std", "pause_ratio"]}
    try:
        import librosa
        y, sr = librosa.load(str(wav_path), sr=None, mono=True)

        # Probabilistic YIN pitch estimation
        f0, voiced, _ = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
        )
        f0_v = f0[voiced] if voiced is not None else f0[~np.isnan(f0)]
        if len(f0_v) == 0:
            f0_v = np.array([0.0])

        # Energy (RMS per frame)
        rms = librosa.feature.rms(y=y)[0]

        # Onset density (speaking rate proxy)
        onsets   = librosa.onset.onset_detect(y=y, sr=sr, units="time")
        duration = librosa.get_duration(y=y, sr=sr)

        return {
            "pitch_mean_hz" : round(float(np.nanmean(f0_v)),  2),
            "pitch_std_hz"  : round(float(np.nanstd(f0_v)),   2),
            "pitch_range_hz": round(float(np.nanmax(f0_v) - np.nanmin(f0_v)), 2),
            "speaking_rate" : round(float(len(onsets) / max(duration, 1e-9)), 2),
            "energy_std"    : round(float(np.std(rms)),   5),
            "pause_ratio"   : round(float(np.mean(rms < 0.01)), 4),
        }
    except Exception as exc:
        log.warning(f"    Prosody extraction failed ({wav_path.name}): {exc}")
        return nan_dict

print("compute_prosody() defined.")

### 6.2 Intelligibility — Whisper ASR → WER / CER

We transcribe each synthesised WAV with **OpenAI Whisper** (`base` model) and compare the transcript to the original input text using `jiwer`.

- **WER (Word Error Rate)** — fraction of words wrong; the standard metric
- **CER (Character Error Rate)** — more sensitive for morphologically rich languages (Hindi)

A perfect TTS that reads the input exactly as written would yield WER ≈ 0, CER ≈ 0.

In [ ]:
_whisper_model_cache = None

def _get_whisper():
    global _whisper_model_cache
    if _whisper_model_cache is None:
        import whisper
        log.info("  [Whisper] Loading base model …")
        _whisper_model_cache = whisper.load_model("base", device=DEVICE)
    return _whisper_model_cache


def compute_intelligibility(
    wav_path: Path, ref_text: str, lang: str
) -> Dict[str, float]:
    """
    Transcribe WAV with Whisper and compute WER and CER against the
    original input text.  Both metrics are clamped at 200% to avoid
    outliers from hallucinated long outputs.
    """
    nan_dict = {"wer": float("nan"), "cer": float("nan")}
    if SKIP_WHISPER:
        return nan_dict
    try:
        from jiwer import cer as jiwer_cer, wer as jiwer_wer
        wmodel    = _get_whisper()
        lang_code = "hi" if lang == "hi" else "en"
        result    = wmodel.transcribe(str(wav_path), language=lang_code)
        hyp       = result["text"].strip()
        ref       = ref_text.strip()
        return {
            "wer": round(min(jiwer_wer(ref, hyp) * 100, 200.0), 2),
            "cer": round(min(jiwer_cer(ref, hyp) * 100, 200.0), 2),
        }
    except ImportError as e:
        log.warning(f"    Intelligibility skipped (missing: {e})")
        return nan_dict
    except Exception as exc:
        log.warning(f"    Intelligibility failed ({wav_path.name}): {exc}")
        return nan_dict

print("compute_intelligibility() defined.")

### 6.3 MOS Prediction — UTMOS

[UTMOS](https://github.com/sarulab-speech/UTMOS22) (Sarulab, 2022) is a neural MOS predictor trained on the VoiceMOS Challenge data.  It predicts a **Mean Opinion Score** in [1, 5] for any WAV file without human listeners.

Install: `pip install utmos`  
If UTMOS is not installed, this function returns `NaN` silently — the rest of the benchmark continues normally.

In [ ]:
_utmos_cache    = None
_utmos_available: Optional[bool] = None

def compute_mos(wav_path: Path) -> float:
    """
    UTMOS neural MOS predictor.
    Returns a score in [1, 5] — higher is better.
    Returns NaN gracefully if UTMOS is not installed.
    """
    global _utmos_cache, _utmos_available
    if SKIP_MOS or _utmos_available is False:
        return float("nan")
    try:
        if _utmos_cache is None:
            import utmos
            _utmos_cache     = utmos.UTMOSScore(device=DEVICE)
            _utmos_available = True
        score = _utmos_cache.score(str(wav_path))
        return round(float(score), 3)
    except ImportError:
        if _utmos_available is None:
            log.warning(
                "  UTMOS not installed — MOS column will be NaN.\n"
                "  Install with: pip install utmos"
            )
        _utmos_available = False
        return float("nan")
    except Exception as exc:
        log.warning(f"    UTMOS failed ({wav_path.name}): {exc}")
        return float("nan")

print("compute_mos() defined.")

## 7. Benchmark Runner

### 7.1 Timed Synthesis

Each utterance is synthesised `N_WARMUP_RUNS + N_TIMED_RUNS` times.  Warm-up runs flush JIT and file-cache cold-start effects.  **Latency = mean wall-clock time** over the timed runs.

In [ ]:
def _timed_synthesis(
    model: BaseTTS,
    text: str,
    lang: str,
    out_path: Path,
) -> Tuple[float, float]:
    """
    Run warm-up + timed synthesis iterations.

    Returns
    -------
    (mean_elapsed_seconds, audio_duration_seconds)
    """
    for _ in range(N_WARMUP_RUNS):
        model.synthesize(text, lang, out_path)

    times, dur = [], 0.0
    for _ in range(N_TIMED_RUNS):
        t0  = time.perf_counter()
        dur = model.synthesize(text, lang, out_path)
        times.append(time.perf_counter() - t0)

    return float(np.mean(times)), dur

print("_timed_synthesis() defined.")

### 7.2 Single-Model Benchmark Loop

In [ ]:
def benchmark_one_model(
    model: BaseTTS, languages: List[str]
) -> List[TTSResult]:
    """
    Synthesise every sentence in every requested language for `model`,
    compute all metrics, and return a list of TTSResult records.
    """
    results: List[TTSResult] = []

    for lang in languages:
        if lang not in model.supported_langs:
            log.info(f"  [{model.name}] '{lang}' not supported — skipping.")
            continue

        corpus = CORPORA[lang]
        log.info(
            f"  [{model.name}][{lang.upper()}] "
            f"Starting {len(corpus)} sentences …"
        )

        for category, text in corpus.items():
            res = TTSResult(
                model_name=model.name, language=lang,
                category=category, text=text,
            )
            safe_name = f"{model.name}_{lang}_{category}.wav".replace("/", "-")
            out_path  = AUDIO_DIR / safe_name

            try:
                elapsed, audio_dur = _timed_synthesis(model, text, lang, out_path)

                res.audio_path        = str(out_path)
                res.audio_duration_s  = round(audio_dur, 3)
                res.latency_ms        = round(elapsed * 1_000, 2)
                res.rtf               = round(elapsed / max(audio_dur, 1e-9), 4)
                res.throughput_cps    = round(len(text) / max(elapsed, 1e-9), 2)

                # Quality
                res.mos_utmos = compute_mos(out_path)
                intel         = compute_intelligibility(out_path, text, lang)
                res.wer       = intel["wer"]
                res.cer       = intel["cer"]

                # Prosody
                pro                = compute_prosody(out_path)
                res.pitch_mean_hz  = pro["pitch_mean_hz"]
                res.pitch_std_hz   = pro["pitch_std_hz"]
                res.pitch_range_hz = pro["pitch_range_hz"]
                res.speaking_rate  = pro["speaking_rate"]
                res.energy_std     = pro["energy_std"]
                res.pause_ratio    = pro["pause_ratio"]

                log.info(
                    f"    [{category:16s}] "
                    f"latency={res.latency_ms:7.1f}ms  "
                    f"RTF={res.rtf:.3f}  "
                    f"WER={res.wer:.1f}%  "
                    f"MOS={res.mos_utmos:.2f}"
                )

            except Exception as exc:
                log.error(f"    [{category}] FAILED: {exc}")
                res.error = str(exc)

            results.append(res)

    return results

print("benchmark_one_model() defined.")

### 7.3 Full Benchmark Orchestrator + CSV Export

In [ ]:
_NUMERIC_COLS = [
    "latency_ms", "rtf", "throughput_cps", "audio_duration_s",
    "mos_utmos", "wer", "cer",
    "pitch_mean_hz", "pitch_std_hz", "pitch_range_hz",
    "speaking_rate", "energy_std", "pause_ratio",
]


def run_benchmark(
    model_names: Optional[List[str]] = None,
    languages: Optional[List[str]]   = None,
) -> pd.DataFrame:
    """
    Run the full benchmark pipeline:
      1. Load each model
      2. Synthesise all corpus sentences + measure all metrics
      3. Unload the model (free memory)
      4. Save 4 CSVs
      5. Return the full results DataFrame
    """
    langs = languages or ["en", "hi"]
    models = (
        [MODEL_REGISTRY[n] for n in model_names if n in MODEL_REGISTRY]
        if model_names else ALL_MODELS
    )
    if not models:
        log.error("No valid models selected.")
        return pd.DataFrame()

    all_results: List[TTSResult] = []

    for model in models:
        sep = "=" * 66
        log.info(f"\n{sep}")
        log.info(f"  Benchmarking:  {model.name}")
        log.info(sep)
        try:
            model.load()
            results = benchmark_one_model(model, langs)
            all_results.extend(results)
        except Exception as exc:
            log.error(f"  [{model.name}] Fatal error during benchmark: {exc}")
        finally:
            try:
                model.unload()
            except Exception:
                pass

    if not all_results:
        log.warning("No results collected.")
        return pd.DataFrame()

    df = pd.DataFrame([asdict(r) for r in all_results])

    # ── CSV 1: full row-per-utterance results ─────────────────────────────────
    p = CSV_DIR / "tts_benchmark_full.csv"
    df.to_csv(p, index=False, encoding="utf-8")
    log.info(f"\n  [CSV] Full results     → {p}")

    # ── CSV 2: per-model-language summary (means) ─────────────────────────────
    p = CSV_DIR / "tts_benchmark_summary.csv"
    (
        df.groupby(["model_name", "language"])[_NUMERIC_COLS]
        .mean(numeric_only=True).round(3).reset_index()
        .to_csv(p, index=False, encoding="utf-8")
    )
    log.info(f"  [CSV] Summary          → {p}")

    # ── CSV 3: linguistic robustness (WER / CER per category) ─────────────────
    p = CSV_DIR / "tts_benchmark_robustness.csv"
    df[["model_name", "language", "category", "wer", "cer"]].dropna(
        subset=["wer"]
    ).to_csv(p, index=False, encoding="utf-8")
    log.info(f"  [CSV] Robustness       → {p}")

    # ── CSV 4: per-model × per-category means ─────────────────────────────────
    p = CSV_DIR / "tts_benchmark_per_model.csv"
    (
        df.groupby(["model_name", "category"])[_NUMERIC_COLS]
        .mean(numeric_only=True).round(3).reset_index()
        .to_csv(p, index=False, encoding="utf-8")
    )
    log.info(f"  [CSV] Per-model/cat    → {p}")

    return df

print("run_benchmark() defined.")

## 8. ▶️ Run the Benchmark

> **Expected runtime:** ~5–30 min on CPU depending on models selected.  With a GPU (T4 or better on Kaggle) the whole suite runs in ~5 min.
>
> To run a quick sanity-check, set `RUN_MODELS = ["MMS-TTS"]` in Section 2.3 above.

In [ ]:
df = run_benchmark(model_names=RUN_MODELS, languages=LANGUAGES)
print(f"\nDataFrame shape: {df.shape}")
df.head()

## 9. Aggregate Results & Console Summary

In [ ]:
summary_cols = ["latency_ms", "rtf", "mos_utmos", "wer", "cer",
                "pitch_std_hz", "energy_std"]

for lang in df["language"].unique():
    print(f"\n{'='*75}")
    print(f"  BENCHMARK SUMMARY — {lang.upper()}")
    print(f"{'='*75}")
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[summary_cols]
        .mean(numeric_only=True)
        .round(3)
    )
    print(sub.to_string())

print("\nNote:  latency_ms↓  rtf↓  mos_utmos↑  wer↓  cer↓  pitch_std_hz↑  energy_std↑")

## 10. Visualisations

11 publication-quality plots are generated and saved to `tts_benchmark_results/plots/`.  Each plot is displayed inline and saved as a 150 DPI PNG.

### 10.0 Style Setup

In [ ]:
_PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2",
            "#937860", "#DA8BC3", "#8C8C8C", "#CCB974", "#64B5CD"]
_MODEL_CLR: Dict[str, str] = {}

def _setup_style():
    global _MODEL_CLR
    models = df["model_name"].unique().tolist()
    _MODEL_CLR = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(models)}
    plt.rcParams.update({
        "figure.facecolor" : "white",
        "axes.facecolor"   : "#f8f9fa",
        "axes.grid"        : True,
        "grid.alpha"       : 0.35,
        "grid.color"       : "#cccccc",
        "font.family"      : "DejaVu Sans",
        "axes.spines.top"  : False,
        "axes.spines.right": False,
    })

def _save(fig, name: str):
    path = PLOT_DIR / f"{name}.png"
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    print(f"  Saved → {path}")
    plt.show()

def _annotate_bars(ax, bars, vals, fmt=".2f"):
    for bar, v in zip(bars, vals):
        if not (isinstance(v, float) and np.isnan(v)):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                f"{v:{fmt}}",
                ha="center", va="bottom", fontsize=8, fontweight="bold",
            )

_setup_style()
print(f"Model colour map: {_MODEL_CLR}")

### 10.1 Performance Metrics — Latency, RTF, Throughput

Bar charts comparing mean latency (ms), Real-Time Factor, and throughput (characters/sec) across models for each language.

- **RTF < 1** = model synthesises faster than real-time (ideal for a live pipeline)
- **Throughput** directly determines pipeline capacity in a batch server setting

In [ ]:
perf_metrics = [
    ("latency_ms",     "Latency (ms)\n↓ lower is better"),
    ("rtf",            "Real-Time Factor\n↓ lower is better  (<1 = faster than real-time)"),
    ("throughput_cps", "Throughput (chars/sec)\n↑ higher is better"),
]

for lang in df["language"].unique():
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[["latency_ms", "rtf", "throughput_cps"]]
        .mean(numeric_only=True).reset_index()
    )
    models = sub["model_name"].tolist()
    colors = [_MODEL_CLR.get(m, "#888") for m in models]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Performance Metrics — {lang.upper()}",
                 fontsize=14, fontweight="bold")

    for ax, (col, title) in zip(axes, perf_metrics):
        vals = sub[col].tolist()
        bars = ax.bar(range(len(models)), vals, color=colors,
                      edgecolor="white", width=0.6)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=22, ha="right", fontsize=9)
        _annotate_bars(ax, bars, vals)

    plt.tight_layout()
    _save(fig, f"01_performance_{lang}")

### 10.2 Quality Metrics — MOS, WER, CER

In [ ]:
qual_metrics = [
    ("mos_utmos", "MOS (UTMOS)\n↑ higher is better  [1–5 scale]"),
    ("wer",       "Word Error Rate (%)\n↓ lower is better"),
    ("cer",       "Character Error Rate (%)\n↓ lower is better"),
]

for lang in df["language"].unique():
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[["mos_utmos", "wer", "cer"]]
        .mean(numeric_only=True).reset_index()
    )
    models = sub["model_name"].tolist()
    colors = [_MODEL_CLR.get(m, "#888") for m in models]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Quality Metrics — {lang.upper()}",
                 fontsize=14, fontweight="bold")

    for ax, (col, title) in zip(axes, qual_metrics):
        vals = sub[col].tolist()
        bars = ax.bar(range(len(models)), vals, color=colors,
                      edgecolor="white", width=0.6)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=22, ha="right", fontsize=9)
        _annotate_bars(ax, bars, vals)

    plt.tight_layout()
    _save(fig, f"02_quality_{lang}")

### 10.3 Prosody Metrics — Pitch, Speaking Rate, Energy, Pauses

In [ ]:
prosody_metrics = [
    ("pitch_mean_hz",  "Mean Pitch (Hz)\nFundamental frequency of voice"),
    ("pitch_std_hz",   "Pitch Std Dev (Hz)\n↑ higher = more varied / natural"),
    ("pitch_range_hz", "Pitch Range (Hz)\nMax − min F0 in voiced frames"),
    ("speaking_rate",  "Speaking Rate (onsets/s)\nProxy for tempo"),
    ("energy_std",     "Energy Dynamics (RMS std)\n↑ higher = more expressive"),
    ("pause_ratio",    "Pause Ratio\nFraction of silent frames"),
]

pros_cols = [c for c, _ in prosody_metrics]

for lang in df["language"].unique():
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[pros_cols]
        .mean(numeric_only=True).reset_index()
    )
    models = sub["model_name"].tolist()
    colors = [_MODEL_CLR.get(m, "#888") for m in models]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Prosody Metrics — {lang.upper()}",
                 fontsize=14, fontweight="bold")

    for ax, (col, title) in zip(axes.flat, prosody_metrics):
        vals = sub[col].tolist()
        bars = ax.bar(range(len(models)), vals, color=colors,
                      edgecolor="white", width=0.6)
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=22, ha="right", fontsize=9)
        _annotate_bars(ax, bars, vals)

    plt.tight_layout()
    _save(fig, f"03_prosody_{lang}")

### 10.4 Linguistic Robustness Heatmap

WER (%) for each model × sentence category combination.  **Green = lower WER = better.**  This reveals which models struggle with specific linguistic phenomena (numbers, named entities, technical terms, punctuation).

In [ ]:
for lang in df["language"].unique():
    sub = df[(df["language"] == lang)].dropna(subset=["wer"])
    if sub.empty:
        print(f"  No WER data for {lang} — skipping heatmap.")
        continue

    pivot = sub.pivot_table(
        index="model_name", columns="category",
        values="wer", aggfunc="mean"
    )
    fig, ax = plt.subplots(
        figsize=(max(10, len(pivot.columns) * 1.6), len(pivot) + 2)
    )
    sns.heatmap(
        pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
        linewidths=0.5, ax=ax,
        cbar_kws={"label": "WER (%) — green = lower = better"},
    )
    ax.set_title(
        f"Linguistic Robustness — WER (%) per Category\nLanguage: {lang.upper()}",
        fontsize=13, fontweight="bold",
    )
    ax.set_xlabel("Sentence Category", fontsize=10)
    ax.set_ylabel("Model",            fontsize=10)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    _save(fig, f"04_robustness_heatmap_{lang}")

### 10.5 WER Grouped Bar Chart by Category

In [ ]:
for lang in df["language"].unique():
    sub = df[(df["language"] == lang)].dropna(subset=["wer"])
    if sub.empty:
        continue

    categories = sorted(sub["category"].unique())
    models     = sub["model_name"].unique()
    x          = np.arange(len(categories))
    n          = len(models)
    width      = 0.8 / n

    fig, ax = plt.subplots(figsize=(max(12, len(categories) * 2.2), 6))

    for i, model in enumerate(models):
        vals   = [sub[(sub["model_name"] == model) &
                      (sub["category"]   == cat)]["wer"].mean()
                  for cat in categories]
        offset = (i - n / 2 + 0.5) * width
        ax.bar(x + offset, vals, width=width * 0.9,
               label=model, color=_MODEL_CLR.get(model, "#888"),
               edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(categories, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("WER (%) — lower is better", fontsize=10)
    ax.set_title(
        f"Word Error Rate by Linguistic Category — {lang.upper()}",
        fontsize=12, fontweight="bold",
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    _save(fig, f"05_wer_by_category_{lang}")

### 10.6 Speed vs Quality Scatter (RTF vs MOS)

Each point is a model.  The **red dashed line** marks RTF = 1.0 (real-time boundary).  The **grey dashed line** marks MOS = 3.5 ("good" quality threshold).  Ideal models sit in the **bottom-right quadrant** (fast + high quality).

In [ ]:
langs = df["language"].unique()
fig, axes = plt.subplots(1, len(langs), figsize=(8 * len(langs), 6))
if len(langs) == 1:
    axes = [axes]

for ax, lang in zip(axes, langs):
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[["rtf", "mos_utmos"]]
        .mean(numeric_only=True).reset_index()
    )
    for _, row in sub.iterrows():
        c = _MODEL_CLR.get(row["model_name"], "#888")
        ax.scatter(row["rtf"], row["mos_utmos"], s=220, color=c, zorder=5)
        ax.annotate(
            row["model_name"], (row["rtf"], row["mos_utmos"]),
            textcoords="offset points", xytext=(8, 5), fontsize=9,
        )
    ax.axhline(3.5, color="gray", ls="--", alpha=0.5, lw=1,
               label="MOS = 3.5 (good quality)")
    ax.axvline(1.0, color="red",  ls="--", alpha=0.5, lw=1,
               label="RTF = 1.0 (real-time boundary)")
    ax.set_xlabel("Real-Time Factor (RTF) — ↓ faster", fontsize=10)
    ax.set_ylabel("MOS (UTMOS) — ↑ better",            fontsize=10)
    ax.set_title(f"Speed–Quality Trade-off — {lang.upper()}",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8)

plt.tight_layout()
_save(fig, "06_speed_vs_quality_scatter")

### 10.7 Latency Distribution — Violin Plot

Shows the **distribution** of latency across all sentence types for each model.  A narrow violin indicates consistent latency; a wide violin indicates variance across sentence lengths / complexities.

In [ ]:
for lang in df["language"].unique():
    sub = df[(df["language"] == lang) & df["latency_ms"].notna()]
    if sub.empty:
        continue

    models    = sub["model_name"].unique()
    data      = [sub[sub["model_name"] == m]["latency_ms"].values for m in models]
    positions = np.arange(len(models))

    fig, ax = plt.subplots(figsize=(12, 6))
    parts = ax.violinplot(data, positions=positions,
                          showmeans=True, showmedians=True, showextrema=True)

    for pc, model in zip(parts["bodies"], models):
        pc.set_facecolor(_MODEL_CLR.get(model, "#888"))
        pc.set_alpha(0.7)

    ax.set_xticks(positions)
    ax.set_xticklabels(models, fontsize=9)
    ax.set_xlabel("Model",        fontsize=10)
    ax.set_ylabel("Latency (ms)", fontsize=10)
    ax.set_title(
        f"Latency Distribution across Sentence Types — {lang.upper()}\n"
        "(violin = density  |  line = median  |  dot = mean)",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    _save(fig, f"07_latency_violin_{lang}")

### 10.8 Radar / Spider Chart — Model Profile

Each model is represented as a polygon over six normalised axes (outer rim = best in class).  Larger area = better overall performance.  Use this to spot model trade-offs at a glance.

In [ ]:
radar_cfg = [
    # (column, label, higher_is_better)
    ("mos_utmos",     "MOS↑",          True),
    ("wer",           "WER↓",          False),
    ("rtf",           "RTF↓",          False),
    ("pitch_std_hz",  "Pitch Var.↑",   True),
    ("throughput_cps","Throughput↑",   True),
    ("energy_std",    "Energy Dyn.↑",  True),
]
r_cols, r_labels, r_hib = zip(*radar_cfg)
n_r  = len(r_cols)
angles = np.linspace(0, 2 * np.pi, n_r, endpoint=False).tolist()
angles += angles[:1]

for lang in df["language"].unique():
    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[list(r_cols)]
        .mean(numeric_only=True).reset_index().dropna()
    )
    if sub.empty:
        continue

    # Normalise to [0, 1] with direction correction
    norm = sub[list(r_cols)].copy()
    for col, hib in zip(r_cols, r_hib):
        rng = norm[col].max() - norm[col].min()
        norm[col] = (norm[col] - norm[col].min()) / rng if rng > 0 else 0.5
        if not hib:
            norm[col] = 1 - norm[col]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    for (_, raw_row), (_, norm_row) in zip(sub.iterrows(), norm.iterrows()):
        model = raw_row["model_name"]
        vals  = norm_row[list(r_cols)].tolist() + [norm_row[r_cols[0]]]
        clr   = _MODEL_CLR.get(model, "#888")
        ax.plot(angles, vals, "o-", linewidth=2, label=model, color=clr)
        ax.fill(angles, vals, alpha=0.07, color=clr)

    ax.set_thetagrids(np.degrees(angles[:-1]), r_labels, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_title(
        f"Model Profile Radar — {lang.upper()}\n"
        "(normalised; outer rim = best in class)",
        fontsize=12, fontweight="bold", pad=28,
    )
    ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.2), fontsize=9)
    plt.tight_layout()
    _save(fig, f"08_radar_{lang}")

### 10.9 Comprehensive Comparison Heatmap

All 10 metrics in one view.  **Cell colour** = normalised rank (green = better).  **Cell number** = raw value.  Use this as the primary one-page summary for selecting a TTS model for your pipeline.

In [ ]:
comp_cfg = [
    # (column, display_label, higher_is_better)
    ("latency_ms",     "Latency↓",     False),
    ("rtf",            "RTF↓",         False),
    ("throughput_cps", "Throughput↑",  True),
    ("mos_utmos",      "MOS↑",         True),
    ("wer",            "WER%↓",        False),
    ("cer",            "CER%↓",        False),
    ("pitch_std_hz",   "Pitch Var.↑",  True),
    ("speaking_rate",  "Speak.Rate",   True),
    ("energy_std",     "Energy Dyn.↑", True),
    ("pause_ratio",    "Pause Ratio↓", False),
]

for lang in df["language"].unique():
    cols = [c for c, _, _ in comp_cfg]
    sub  = (
        df[df["language"] == lang]
        .groupby("model_name")[cols]
        .mean(numeric_only=True).reset_index().set_index("model_name")
    )
    rename_map = {c: lbl for c, lbl, _ in comp_cfg}
    hib_map    = {lbl: hib for _, lbl, hib in comp_cfg}
    sub.rename(columns=rename_map, inplace=True)

    norm = sub.copy()
    for col in norm.columns:
        rng = norm[col].max() - norm[col].min()
        norm[col] = (norm[col] - norm[col].min()) / rng if rng > 0 else 0.5
        if not hib_map.get(col, True):
            norm[col] = 1 - norm[col]

    fig, ax = plt.subplots(
        figsize=(len(sub.columns) * 1.4 + 2, len(sub) + 2)
    )
    sns.heatmap(
        norm, annot=sub.round(2), fmt="g",
        cmap="RdYlGn", linewidths=0.5, ax=ax,
        cbar_kws={"label": "Normalised score (green = better)"},
        vmin=0, vmax=1,
    )
    ax.set_title(
        f"Comprehensive Model Comparison — {lang.upper()}\n"
        "Colour = normalised rank  |  Number = raw value",
        fontsize=12, fontweight="bold",
    )
    ax.set_ylabel("Model", fontsize=10)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    _save(fig, f"09_comprehensive_heatmap_{lang}")

### 10.10 Audio Duration vs Synthesis Latency Scatter

Points **below** the red RTF = 1.0 line are synthesised faster than real-time — a hard requirement for a low-latency speech-to-speech pipeline.

In [ ]:
for lang in df["language"].unique():
    sub = df[
        (df["language"] == lang) &
        df["audio_duration_s"].notna() &
        df["latency_ms"].notna()
    ]
    if sub.empty:
        continue

    fig, ax = plt.subplots(figsize=(9, 6))

    for model, grp in sub.groupby("model_name"):
        ax.scatter(
            grp["audio_duration_s"], grp["latency_ms"],
            label=model, color=_MODEL_CLR.get(model, "#888"),
            s=70, alpha=0.8,
        )

    max_dur = sub["audio_duration_s"].max()
    ax.plot([0, max_dur], [0, max_dur * 1_000],
            "r--", lw=1.5, alpha=0.6, label="RTF = 1.0 (real-time)")

    ax.set_xlabel("Audio Duration (s)",         fontsize=10)
    ax.set_ylabel("Synthesis Latency (ms)",     fontsize=10)
    ax.set_title(
        f"Audio Duration vs Synthesis Latency — {lang.upper()}\n"
        "Points below the red line are synthesised faster than real-time",
        fontsize=11, fontweight="bold",
    )
    ax.legend(fontsize=9)
    plt.tight_layout()
    _save(fig, f"10_duration_vs_latency_{lang}")

### 10.11 Pitch Distribution — Box Plots

In [ ]:
for lang in df["language"].unique():
    sub = df[(df["language"] == lang) & df["pitch_mean_hz"].notna()]
    if sub.empty:
        continue

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Pitch Distribution — {lang.upper()}",
                 fontsize=13, fontweight="bold")

    for ax, (col, title) in zip(axes, [
        ("pitch_mean_hz", "Mean Pitch (Hz)"),
        ("pitch_std_hz",  "Pitch Std Dev (Hz) — naturalness"),
    ]):
        models = list(sub["model_name"].unique())
        data   = [sub[sub["model_name"] == m][col].values for m in models]
        bps = ax.boxplot(data, patch_artist=True, labels=models)
        for patch, model in zip(bps["boxes"], models):
            patch.set_facecolor(_MODEL_CLR.get(model, "#888"))
            patch.set_alpha(0.75)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xticklabels(models, rotation=22, ha="right", fontsize=9)

    plt.tight_layout()
    _save(fig, f"11_pitch_boxplot_{lang}")

## 11. Final Rankings

Rank models on each metric (1 = best).  Lower **Avg Rank** = better overall.  The overall winner is the recommended model for each language leg of your S2S pipeline.

In [ ]:
rank_metrics = ["latency_ms", "rtf", "mos_utmos", "wer", "cer",
                "pitch_std_hz", "energy_std", "throughput_cps"]

# higher_is_better columns
_hib_set = {"mos_utmos", "pitch_std_hz", "energy_std", "throughput_cps"}

for lang in df["language"].unique():
    print(f"\n{'='*70}")
    print(f"  FINAL RANKING — {lang.upper()}")
    print(f"{'='*70}")

    sub = (
        df[df["language"] == lang]
        .groupby("model_name")[rank_metrics]
        .mean(numeric_only=True)
        .dropna(how="all")
    )

    print("\nRaw means:")
    print(sub.round(3).to_string())

    rank_df = sub.copy()
    for col in rank_df.columns:
        asc = col not in _hib_set   # ascending = lower is better
        rank_df[col] = rank_df[col].rank(ascending=asc).astype(int)
    rank_df["Avg Rank"] = rank_df.mean(axis=1).round(2)

    print("\nRankings (1 = best per metric):")
    print(rank_df.to_string())

    winner = rank_df["Avg Rank"].idxmin()
    print(f"\n  ★  Overall winner ({lang.upper()}): {winner}  "
          f"(avg rank = {rank_df.loc[winner, 'Avg Rank']})")

    # Save ranking CSV
    rank_df.reset_index().to_csv(
        CSV_DIR / f"tts_benchmark_ranking_{lang}.csv",
        index=False, encoding="utf-8"
    )

print(f"\n  Ranking CSVs saved to {CSV_DIR}/")

## 12. Output Summary

In [ ]:
import os

print("\n" + "="*60)
print("  TTS BENCHMARK — OUTPUT FILES")
print("="*60)

print("\n📊 CSVs:")
for f in sorted(CSV_DIR.glob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:6.1f} KB")

print("\n🖼️  Plots:")
for f in sorted(PLOT_DIR.glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:6.1f} KB")

print("\n🔊 Audio WAVs:")
wavs = list(AUDIO_DIR.glob("*.wav"))
print(f"  {len(wavs)} WAV files in {AUDIO_DIR}")

print("\n" + "="*60)
print("  Benchmark complete!")
print("="*60)

---

## Appendix — Metric Reference

| Metric | Range | Direction | Notes |
|---|---|---|---|
| **Latency (ms)** | 0 → ∞ | ↓ lower | Wall-clock synthesis time, averaged over `N_TIMED_RUNS` |
| **RTF** | 0 → ∞ | ↓ lower | `synthesis_time / audio_duration`; RTF < 1 = faster than real-time |
| **Throughput (CPS)** | 0 → ∞ | ↑ higher | Characters synthesised per second |
| **MOS (UTMOS)** | 1 → 5 | ↑ higher | Neural MOS predictor; ≥ 3.5 is considered good quality |
| **WER (%)** | 0 → 200 | ↓ lower | Whisper transcription error rate vs original text |
| **CER (%)** | 0 → 200 | ↓ lower | Character-level transcription error; more sensitive for Hindi |
| **Pitch mean (Hz)** | ~80–350 | — | Voice fundamental frequency; reflects speaker identity |
| **Pitch std (Hz)** | 0 → ∞ | ↑ higher | Pitch variation → naturalness and expressiveness |
| **Pitch range (Hz)** | 0 → ∞ | context | Max − min F0; wider range = more expressive |
| **Speaking rate** | 0 → ∞ | context | Onset events/sec; 4–6 ≈ natural conversational pace |
| **Energy std (RMS)** | 0 → 1 | ↑ higher | Higher variance = more dynamic / expressive audio |
| **Pause ratio** | 0 → 1 | context | Fraction of silent frames; too high = unnatural halting speech |

---

## Pipeline Recommendation

For a Hindi ↔ English Speech-to-Speech pipeline, weight the metrics as follows:

| Priority | Metric | Reason |
|---|---|---|
| 1 (critical) | **RTF < 1.0** | Must synthesise faster than real-time for live use |
| 2 (critical) | **WER < 15%** | Listeners must be able to understand the output |
| 3 (important) | **MOS > 3.5** | Poor quality degrades user experience |
| 4 (nice-to-have) | **Pitch std, Energy std** | Natural prosody improves perceived quality |
| 5 (nice-to-have) | **Latency (ms)** | Absolute latency matters less than RTF in batch mode |